# Beer Data Analysis

## Assignment

In this assignment you will work with a beer data set. Please provide an answer to the questions below. Answer as many questions as possible:

1. Rank the top 3 breweries which produce the strongest beers.
2. Which year did beers enjoy the highest ratings?
3. Based on the users' ratings, which factors are important among taste, aroma, appearance, and palette?
4. If you were to recommend 3 beers to your friends based on this data, which ones would you recommend?
5. Which beer style seems to be the favourite based on the reviews written by users? How does written reviews compare to overall review score for the beer style?

In [21]:
import numpy as np
import pandas as pd

In [22]:
data = pd.read_csv('/Users/rosiebai/Downloads/datasets-6/BeerDataScienceProject.tar.bz2', compression="bz2")

In [23]:
data.head()

,beer_ABV,beer_beerId,beer_brewerId,beer_name,beer_style,review_appearance,review_palette,review_overall,review_taste,review_profileName,review_aroma,review_text,review_time
0,5.0,47986,10325,Sausa Weizen,Hefeweizen,2.5,2.0,1.5,1.5,stcules,1.5,A lot of foam. But a lot. In the smell some ba...,1234817823
1,6.2,48213,10325,Red Moon,English Strong Ale,3.0,2.5,3.0,3.0,stcules,3.0,"Dark red color, light beige foam, average. In ...",1235915097
2,6.5,48215,10325,Black Horse Black Beer,Foreign / Export Stout,3.0,2.5,3.0,3.0,stcules,3.0,"Almost totally black. Beige foam, quite compac...",1235916604
3,5.0,47969,10325,Sausa Pils,German Pilsener,3.5,3.0,3.0,2.5,stcules,3.0,"Golden yellow color. White, compact foam, quit...",1234725145
4,7.7,64883,1075,Cauldron DIPA,American Double / Imperial IPA,4.0,4.5,4.0,4.0,johnmichaelsen,4.5,"According to the website, the style for the Ca...",1293735206


In [24]:
data.dtypes

beer_ABV              float64
beer_beerId             int64
beer_brewerId           int64
beer_name              object
beer_style             object
review_appearance     float64
review_palette        float64
review_overall        float64
review_taste          float64
review_profileName     object
review_aroma          float64
review_text            object
review_time             int64
dtype: object

## 1. Rank the top 3 breweries which produce the strongest beers.


In [25]:
avg_taste_by_brewer = data.groupby('beer_brewerId')['beer_ABV'].mean().reset_index(name = 'avg_beer_ABV').sort_values(by = 'avg_beer_ABV', ascending=False)
avg_taste_by_brewer['rank'] = avg_taste_by_brewer['avg_beer_ABV'].rank(method = 'dense',ascending = False)
avg_taste_by_brewer[avg_taste_by_brewer['rank'] <= 3]

,beer_brewerId,avg_beer_ABV,rank
784,6513,19.228824,1.0
175,736,13.750000,2.0
1644,24215,12.466667,3.0


## 2. Which year did beers enjoy the highest ratings?


In [26]:
data['review_time'] = pd.to_datetime(data['review_time'], unit='s', errors='coerce')
data['review_year'] = data['review_time'].dt.year

In [27]:

data.groupby('review_year')['review_overall'].mean().reset_index(name = 'avg_rating').sort_values(by = 'avg_rating', ascending = False)

,review_year,avg_rating
2,2000,4.181818
1,1999,4.000000
3,2001,3.927741
0,1998,3.891304
12,2010,3.866139
11,2009,3.864390
10,2008,3.833939
7,2005,3.832042
14,2012,3.829717
13,2011,3.828093


In [28]:
data['review_year'].describe()

count    528870.000000
mean       2008.307208
std           2.409739
min        1998.000000
25%        2007.000000
50%        2009.000000
75%        2010.000000
max        2012.000000
Name: review_year, dtype: float64

2000 seems to have the highest beer rating.

## 3. Based on the users' ratings, which factors are important among taste, aroma, appearance, and palette?


In [29]:
data.columns

Index(['beer_ABV', 'beer_beerId', 'beer_brewerId', 'beer_name', 'beer_style',
       'review_appearance', 'review_palette', 'review_overall', 'review_taste',
       'review_profileName', 'review_aroma', 'review_text', 'review_time',
       'review_year'],
      dtype='object')

In [30]:
data.isna().sum()

beer_ABV              20280
beer_beerId               0
beer_brewerId             0
beer_name                 0
beer_style                0
review_appearance         0
review_palette            0
review_overall            0
review_taste              0
review_profileName      115
review_aroma              0
review_text             119
review_time               0
review_year               0
dtype: int64

In [46]:
import statsmodels.api as sm
import pandas as pd

# features
X = data[['review_appearance', 'review_palette', 'review_taste', 'review_aroma']]
y = data['review_overall']

# add intercept (VERY important)
X = sm.add_constant(X)

# fit model
model = sm.OLS(y, X).fit()

# full summary
print(model.summary())
results = pd.DataFrame({
    'feature': model.params.index,
    'coefficient': model.params.values,
    'p_value': model.pvalues.values,
    't_stat': model.tvalues.values
})

print(results)

                            OLS Regression Results                            
Dep. Variable:         review_overall   R-squared:                       0.648
Model:                            OLS   Adj. R-squared:                  0.648
Method:                 Least Squares   F-statistic:                 2.430e+05
Date:                Sat, 02 May 2026   Prob (F-statistic):               0.00
Time:                        14:48:01   Log-Likelihood:            -2.9348e+05
No. Observations:              528870   AIC:                         5.870e+05
Df Residuals:                  528865   BIC:                         5.870e+05
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 0.4483      0.00

Beer aroma turned out to be the most important factor, then it's the taste. And it makes sense, usually before we drink something, we smell it first, and if it smells very good, it might increase our satisfaction with the beer. 

## 4. If you were to recommend 3 beers to your friends based on this data, which ones would you recommend?


In [32]:
data.review_text.isna().sum()/len(data)

0.00022500803600128576

In [ ]:
data2 = data.dropna(subset = ['beer_style', 'review_text', 'review_overall'])

In [ ]:
import re
import nltk 
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
custom_stopwords = {'beer','like','taste','drink','flavor'}
stop_words.update(custom_stopwords)

lemmatizer = WordNetLemmatizer()
def clean_text2(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]','', text)
    tokens = word_tokenize(text)
    return ' '.join(
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    )
data2['clean_review'] = data2['review_text'].apply(clean_text2)

In [ ]:
# sentiment analysis on text 
from nltk.sentiment import SentimentIntensityAnalyzer 
import nltk 
nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer() 
data2['sentiment_score'] = data2['clean_review'].apply(lambda x: sia.polarity_scores(x)['compound'])

In [45]:
avg_rating_aroma_taste = (
    data2.groupby(['beer_name', 'beer_style'])
    .agg({
        'review_overall': 'mean',
        'review_aroma': 'mean',
        'review_taste': 'mean',
        'review_palette': 'mean',
        'review_appearance': 'mean',
        'sentiment_score': 'mean',
        'review_text': 'count'   # count of reviews
    })
    .rename(columns={'review_text': 'review_count'})  # optional but clearer
    .reset_index()
    .sort_values(by=['review_overall', 'review_aroma', 'review_taste', 'review_count','sentiment_score'], ascending=False)
)

avg_rating_aroma_taste['rank'] = avg_rating_aroma_taste[['review_overall']].rank(method = 'min', ascending=False)
staff_pick = avg_rating_aroma_taste[(avg_rating_aroma_taste['rank'] == 1)
                       & (avg_rating_aroma_taste['review_aroma'] >= 4)
                       & (avg_rating_aroma_taste['review_taste'] >= 3)
                       & (avg_rating_aroma_taste['review_palette']> 3)
                       & (avg_rating_aroma_taste['review_appearance']> 3)]
staff_pick

,beer_name,beer_style,review_overall,review_aroma,review_taste,review_palette,review_appearance,sentiment_score,review_count,rank
10143,Lips Of Faith - Eric's Ale (Bourbon Barrel Aged),American Wild Ale,5.0,5.0,5.0,4.75,4.75,0.98465,2,1.0
10303,Love Child Belgiweizen,Belgian Strong Pale Ale,5.0,5.0,5.0,4.75,4.75,0.97185,2,1.0
4831,Date Night With Jumbo Love,American Barleywine,5.0,5.0,5.0,5.00,5.00,0.93565,2,1.0
7129,Great Lakes Truth Justice And The American Ale,American Pale Ale (APA),5.0,5.0,5.0,5.00,5.00,0.99570,1,1.0
2916,Bourbon Barrel Coffee Night Train,American Porter,5.0,5.0,5.0,4.50,4.50,0.99480,1,1.0
...,...,...,...,...,...,...,...,...,...,...
14908,"Shakespeare's ""Plonk""",English Bitter,5.0,4.0,4.0,4.00,4.00,0.65970,1,1.0
18171,White Winter Belgian Ale,Witbier,5.0,4.0,4.0,4.00,3.50,0.52990,1,1.0
6414,Fox Tail Amber Ale,American Amber / Red Ale,5.0,4.0,4.0,4.00,4.00,-0.79640,1,1.0
13395,Quinn's Marathon Mild,English Dark Mild Ale,5.0,4.0,3.5,4.00,4.00,0.97360,1,1.0


If I were to recommend to my friends, I would pick the lager with the highest overall rating, good aroma, good taste and decent appearance and palete, and the beers with more reviews. Hence, my recommendations are Date Night With Jumbo Love, Lips Of Faith - Eric's Ale (Bourbon Barrel Aged), Love Child Belgiweizen.

## 5. Which beer style seems to be the favourite based on the reviews written by users? How does written reviews compare to overall review score for the beer style?

In [38]:
data2.head()

,beer_ABV,beer_beerId,beer_brewerId,beer_name,beer_style,review_appearance,review_palette,review_overall,review_taste,review_profileName,review_aroma,review_text,review_time,review_year,clean_review
0,5.0,47986,10325,Sausa Weizen,Hefeweizen,2.5,2.0,1.5,1.5,stcules,1.5,A lot of foam. But a lot. In the smell some ba...,2009-02-16 20:57:03,2009,lot foam lot smell banana lactic tart good sta...
1,6.2,48213,10325,Red Moon,English Strong Ale,3.0,2.5,3.0,3.0,stcules,3.0,"Dark red color, light beige foam, average. In ...",2009-03-01 13:44:57,2009,dark red color light beige foam average smell ...
2,6.5,48215,10325,Black Horse Black Beer,Foreign / Export Stout,3.0,2.5,3.0,3.0,stcules,3.0,"Almost totally black. Beige foam, quite compac...",2009-03-01 14:10:04,2009,almost totally black beige foam quite compact ...
3,5.0,47969,10325,Sausa Pils,German Pilsener,3.5,3.0,3.0,2.5,stcules,3.0,"Golden yellow color. White, compact foam, quit...",2009-02-15 19:12:25,2009,golden yellow color white compact foam quite c...
4,7.7,64883,1075,Cauldron DIPA,American Double / Imperial IPA,4.0,4.5,4.0,4.0,johnmichaelsen,4.5,"According to the website, the style for the Ca...",2010-12-30 18:53:26,2010,according website style caldera cauldron chang...


In [40]:
# aggregate the sentiment scores by beer style
agg_df = data2.groupby('beer_style').agg({
    'review_overall':'mean',
    'sentiment_score':'mean',
    'review_text':'count'
}).rename(columns = {
    'review_overall':'avg_rating',
    'sentiment_score':'avg_sentiment',
    'review_text':'review_count'
}).sort_values('avg_sentiment',ascending=False)
agg_df

,avg_rating,avg_sentiment,review_count
beer_style,,,
Eisbock,4.079487,0.876705,195
Quadrupel (Quad),4.049159,0.872963,4933
Flanders Red Ale,3.961835,0.860493,2856
American Double / Imperial Stout,4.100441,0.857466,23352
Wheatwine,3.817059,0.857361,891
...,...,...,...
Happoshu,2.818182,0.556395,55
Japanese Rice Lager,3.032258,0.549719,496
American Malt Liquor,2.721986,0.523239,1410


In [41]:
data2[['sentiment_score', 'review_overall']].corr()

,sentiment_score,review_overall
sentiment_score,1.000000,0.346192
review_overall,0.346192,1.000000


Eisbock looks like the favoriate beer style by the reviewers based on the sentiment score, but customer review rating and review text are not highly correlated, the correlations is only 0.34. The beer styles that have high sentiment scores don't necessarily guarantee a high overall rating, but they do seem to be moving toward the same direction, which means that they agree with each sometimes. 

## Topics Modeling for a given beer style

In [42]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Choose a specific style to analyze
ipa_df = data2[data2['beer_style'].str.contains('ipa', case=False, na=False)]

# Vectorize text
vectorizer = CountVectorizer(max_df=0.95, min_df=5, stop_words='english')
doc_term_matrix = vectorizer.fit_transform(ipa_df['clean_review'])

# LDA
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(doc_term_matrix)

# Show top words per topic
words = vectorizer.get_feature_names_out()
for i, topic in enumerate(lda.components_):
    top_words = [words[i] for i in topic.argsort()[-10:]]
    print(f"Topic {i+1}: {' | '.join(top_words)}")


Topic 1: pine | good | really | great | ipa | citrus | head | malt | hop | nice
Topic 2: poured | color | citrus | malt | glass | nice | smell | good | head | hop
Topic 3: citrus | bitterness | alcohol | glass | head | orange | pine | grapefruit | malt | hop
Topic 4: carbonation | nice | citrus | bitterness | aroma | light | finish | head | malt | hop
Topic 5: ive | bottle | head | really | ale | smell | good | im | ipa | hop


In [43]:
# Choose a specific style to analyze 
# Filter rows where beer_style contains 'lager' (case-insensitive)
lager_df = data2[data2['beer_style'].str.contains('lager', case=False, na=False)]

# Vectorize text
vectorizer = CountVectorizer(max_df = 0.95, min_df = 5, stop_words = 'english')
doc_term_matrix = vectorizer.fit_transform(lager_df['clean_review'])

# LDA
lda = LatentDirichletAllocation(n_components = 5, random_state = 123)
lda.fit(doc_term_matrix)
# Show top words per topic
words = vectorizer.get_feature_names_out()
for i, topic in enumerate(lda.components_):
    top_words = [words[i] for i in topic.argsort()[-10:]]
    print(f"Topic {i+1}: {' | '.join(top_words)}")


Topic 1: good | aroma | finish | amber | nice | sweet | caramel | head | hop | malt
Topic 2: bit | smell | corn | malt | white | hop | carbonation | sweet | light | head
Topic 3: really | head | beer | bad | good | better | macro | lager | smell | light
Topic 4: carbonation | color | white | nice | good | lager | malt | head | hop | light
Topic 5: great | im | really | brew | smell | time | beer | bottle | lager | good


- Citrus, hop, malt (IPA characteristics)
- Color, head, carbonation (appearance)
- Sweet, caramel (lager/amber profiles)